# T5 for Abstractive Summarization (Description → Headline)

**T5** (Text-To-Text Transfer Transformer) is a *full* encoder-decoder transformer: the
encoder reads the input sequence and the decoder generates an output sequence one token at a
time. Unlike BERT (encoder-only, no generation), T5 frames *every* task as **text-in, text-out**.

Here we fine-tune T5 to turn a Reuters news **Description** into its **Headline**. We prepend the
task prefix `"summarize: "` to every input, which is the convention T5 was pretrained with.

> **Before you run this:** switch on the GPU with **Runtime → Change runtime type → Hardware
> accelerator → GPU (T4)**. On CPU this will be extremely slow.

In [1]:
# Clone the repo (skips if it already exists)
![ -d /content/Bert-T5-GPT2 ] || git clone https://github.com/nickkats1/Bert-T5-GPT2

Cloning into 'Bert-T5-GPT2'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 75 (delta 17), reused 74 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 3.70 MiB | 40.73 MiB/s, done.
Resolving deltas: 100% (17/17), done.


## Install the package and its dependencies
Installing the repo in editable mode (`-e`) pulls in everything the T5 pipeline needs —
`transformers`, `sentencepiece` (required by the T5 tokenizer), `rouge_score`, and `nltk` — and
makes `import headlines...` work without any `sys.path` hacks.

In [2]:
%pip install -q -e /content/Bert-T5-GPT2

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Building editable for bert-t5-gpt2 (pyproject.toml) ... done


In [5]:
# do this so you can run the code base in colab.
import sys
sys.path.append('/content/Bert-T5-GPT2')

## Confirm the GPU is visible
`resolve_device` falls back to CPU if no GPU is found, so this cell is your sanity check that
Colab actually gave you a CUDA device.

In [6]:
import torch
from headlines.common import resolve_device, seed_everything, count_parameters, make_generator
from headlines.t5.config import CONFIG

seed_everything(CONFIG.seed)          # reproducible splits / shuffling
device = resolve_device(CONFIG.device)
print('Device:', device)
assert device.type == 'cuda', 'No GPU! Enable it via Runtime > Change runtime type.'

Device: cuda:0


## Load the data and add the task prefix
`load_data` reads the Reuters CSV, drops the `Time` column, fills NaNs, de-duplicates, and keeps
the `Description` (source) and `Headlines` (target) columns. We then prepend `CONFIG.source_prefix`
(`"summarize: "`) to each description — this is how T5 is told *which* task to perform.

In [7]:
import pandas as pd
from headlines.t5.utils import load_data

df = load_data('/content/Bert-T5-GPT2/data/reuters_headlines.csv')
df['Description'] = CONFIG.source_prefix + df['Description'].astype(str)
print(f'{len(df):,} rows')
df.head()

32,673 rows


,Headlines,Description
0,TikTok considers London and other locations fo...,summarize: TikTok has been in discussions with...
1,Disney cuts ad spending on Facebook amid growi...,summarize: Walt Disney has become the latest ...
2,Trail of missing Wirecard executive leads to B...,summarize: Former Wirecard chief operating of...
3,Twitter says attackers downloaded data from up...,summarize: Twitter Inc said on Saturday that h...
4,U.S. Republicans seek liability protections as...,summarize: A battle in the U.S. Congress over ...


## Load the tokenizer and the pretrained T5 model
`T5ForConditionalGeneration` is the encoder-decoder model with a language-modeling head, ready to
generate text. We start from the pretrained `t5-base` weights and fine-tune from there.

In [8]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained(CONFIG.model_name)
model = T5ForConditionalGeneration.from_pretrained(CONFIG.model_name).to(device)
print(f'Trainable parameters: {count_parameters(model):,}')

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Trainable parameters: 222,903,552


## Train / validation split and DataLoaders
`SummarizationDataset` tokenizes each (source, target) pair to fixed lengths
(`source_length=128`, `target_length=32`). Each item returns `source_ids`, `source_mask`, and
`target_ids`. We wrap the datasets in `DataLoader`s, seeding the training shuffle with
`make_generator` so the run is reproducible.

In [9]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from headlines.t5.dataset import SummarizationDataset

df_train, df_val = train_test_split(df, test_size=CONFIG.test_size, random_state=CONFIG.seed)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
print(f'train={len(df_train):,}  val={len(df_val):,}')

def make_set(frame):
    return SummarizationDataset(
        frame, tokenizer,
        source_len=CONFIG.source_length,
        target_len=CONFIG.target_length,
        source_col='Description',
        target_col='Headlines',
    )

train_loader = DataLoader(
    make_set(df_train), batch_size=CONFIG.batch_size,
    shuffle=True, generator=make_generator(CONFIG.seed),
)
val_loader = DataLoader(make_set(df_val), batch_size=CONFIG.batch_size, shuffle=False)

train=26,138  val=6,535


### Peek at one batch
You should see `source_ids`, `source_mask`, and `target_ids`, each of shape
`(batch_size, sequence_length)`. If so, the dataset and collation are wired up correctly.

In [10]:
batch = next(iter(train_loader))
{k: tuple(v.shape) for k, v in batch.items()}

{'source_ids': (12, 128), 'source_mask': (12, 128), 'target_ids': (12, 32)}

## Optimizer
A plain `AdamW` over all model parameters is enough for fine-tuning. The learning rate
(`5e-5`) and weight decay come from `CONFIG`.

In [11]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay,
)

## Fine-tune
`train` runs one epoch and returns the mean loss. It passes `target_ids` as `labels` (with pad
tokens masked to `-100` so padding is ignored by the loss) and lets T5 build the decoder inputs
via its internal right-shift — keeping training consistent with generation at inference time.

> `t5-base` for 2 epochs on the full dataset takes a while on a T4. To do a quick smoke-test
> first, set `CONFIG` to fewer epochs / a smaller model with `dataclasses.replace`, e.g.
> `cfg = dataclasses.replace(CONFIG, epochs=1, model_name='t5-small')`.

In [12]:
from headlines.t5.trainer import train

for epoch in range(CONFIG.epochs):
    loss = train(model, train_loader, optimizer, device, tokenizer=tokenizer, epoch=epoch)
    print(f'Epoch {epoch + 1}/{CONFIG.epochs} | mean train loss: {loss:.4f}')

21:40:51 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 0 | Loss: 3.8305
21:40:55 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 50 | Loss: 2.5895
21:40:58 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 100 | Loss: 2.5894
21:41:02 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 150 | Loss: 2.3320
21:41:05 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 200 | Loss: 2.1978
21:41:08 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 250 | Loss: 2.5246
21:41:12 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 300 | Loss: 2.0063
21:41:15 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 350 | Loss: 2.2593
21:41:18 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 400 | Loss: 2.4912
21:41:22 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 450 | Loss: 2.1477
21:41:25 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 500 | Loss: 1.7236
21:41:29 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 550 | Loss: 2.3114
21:41:32 | headlines.t5.trainer | INFO | Epoch: 0 | Step: 600 | Loss: 2.3141
21

Epoch 1/2 | mean train loss: 2.1523


21:43:20 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 50 | Loss: 1.9091
21:43:24 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 100 | Loss: 1.9072
21:43:27 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 150 | Loss: 2.2446
21:43:30 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 200 | Loss: 1.6544
21:43:34 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 250 | Loss: 1.6314
21:43:37 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 300 | Loss: 1.9416
21:43:40 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 350 | Loss: 1.5139
21:43:44 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 400 | Loss: 2.0745
21:43:47 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 450 | Loss: 2.0237
21:43:50 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 500 | Loss: 2.0261
21:43:54 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 550 | Loss: 2.3729
21:43:57 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 600 | Loss: 2.0331
21:44:00 | headlines.t5.trainer | INFO | Epoch: 1 | Step: 650 | Loss: 1.8641


Epoch 2/2 | mean train loss: 1.9035


## Generate headlines and score them
`validate` runs beam-search generation over the validation loader and returns the decoded
`(predictions, actuals)`. `compute_metrics` then reports ROUGE-1/2/L, METEOR, and the mean
generation length — the standard lexical/semantic overlap metrics for summarization.

In [13]:
from headlines.t5.trainer import validate
from headlines.t5.metrics import compute_metrics

predictions, actuals = validate(model, val_loader, device, tokenizer=tokenizer)
metrics = compute_metrics(predictions, actuals)
{k: round(v, 4) for k, v in metrics.items()}

21:46:01 | headlines.t5.trainer | INFO | Validation step: 0
21:46:03 | headlines.t5.trainer | INFO | Validation step: 10
21:46:05 | headlines.t5.trainer | INFO | Validation step: 20
21:46:08 | headlines.t5.trainer | INFO | Validation step: 30
21:46:10 | headlines.t5.trainer | INFO | Validation step: 40
21:46:12 | headlines.t5.trainer | INFO | Validation step: 50
21:46:14 | headlines.t5.trainer | INFO | Validation step: 60
21:46:16 | headlines.t5.trainer | INFO | Validation step: 70
21:46:19 | headlines.t5.trainer | INFO | Validation step: 80
21:46:21 | headlines.t5.trainer | INFO | Validation step: 90
21:46:23 | headlines.t5.trainer | INFO | Validation step: 100
21:46:25 | headlines.t5.trainer | INFO | Validation step: 110
21:46:27 | headlines.t5.trainer | INFO | Validation step: 120
21:46:30 | headlines.t5.trainer | INFO | Validation step: 130
21:46:32 | headlines.t5.trainer | INFO | Validation step: 140
21:46:34 | headlines.t5.trainer | INFO | Validation step: 150
21:46:36 | headline

{'rouge1': 0.4837,
 'rouge2': 0.2409,
 'rougeL': 0.4398,
 'meteor': 0.3636,
 'gen_len': 9.8569}

### Eyeball a few predictions
Metrics only tell you so much — read a handful of generated vs. real headlines to judge quality.

In [14]:
pd.DataFrame({'Generated': predictions[:10], 'Actual': actuals[:10]})

,Generated,Actual
0,IAG CEO rules out bidding for Thomas Cook's ai...,"Virgin Atlantic in, IAG out of race for Thomas..."
1,China's Wuhan considers supporting Dongfeng in...,"Exclusive: Wuhan, recovering from virus, weigh..."
2,Tesla to buy energy storage company Maxwell fo...,Tesla to buy battery tech maker Maxwell Techno...
3,Lockheed Martin says coronavirus delays shipme...,Lockheed Martin trims sales outlook as coronav...
4,"SoftBank shares fall 17% on tech bets, skepticism",SoftBank shares close down 17% in biggest one-...
5,Russia tycoon Deripaska gives evidence in Lond...,Russian billionaires take boardroom battle to ...
6,"Global stocks rise ahead of G20, dollar holds ...","Stocks gain on U.S.-China trade hopes, dollar ..."
7,WeWork's May 2025 bond falls after parent post...,"Wework 7.875% bond falls 96.3 cents, lowest si..."
8,Trump says Xi conversation has had impact on U...,Trump touts effect of his call with China's Xi...
9,Anta Sports consortium makes offer to buy Finl...,China's Anta Sports-led group buying Finland's...


## Do not Fine-Tune T5 for Summarization this way.
No one really needs to do it and it takes so long. If you do need to fine-tune it, use the `transformers` library with `TrainingArguments`. Or use the `peft` library. Do not fine-tune a full LLM. T5, since it is seq-seq, is technically a llm.